# **Actividad: DASHboards y Plotly**

#  Pokémon Complete Pokédex — Análisis Exploratorio Visual

**Dataset:** [Pokémon Complete Pokédex Dataset (2026) — Kaggle](https://www.kaggle.com/datasets/patelris/pokemon-dataset-with-stats-and-types)

## Descripción del Dataset

Este dataset contiene información completa de los **1,350 Pokémon** registrados hasta 2026. Incluye:

- **Stats de combate:** HP, Ataque, Defensa, Ataque Especial, Defensa Especial, Velocidad y Stat Total Base
- **Clasificación:** Tipo primario y secundario, generación, color, forma
- **Metadatos:** Tasa de captura, felicidad base, experiencia base, grupos huevo
- **Flags especiales:** `is_legendary`, `is_mythical`, `is_baby`
- **Información adicional:** Habitat, habilidades, cadena de evolución, sprite URL

### ¿Por qué es interesante?
Pokémon es una franquicia con 30 años de historia y diseño de juego profundo.
Analizar sus datos permite descubrir patrones de balance, tendencias por generación
y diferencias estratégicas entre tipos, lo que lo convierte en un dataset
rico para visualización y exploración de datos.

---

**Integrantes del equipo:**
- Jorge Emiliano Gómez Hernández | A01368016
- Jaime Trujillo Ruiz | A01276139



## Esquemas de color

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

paletas = {
    'Análogos':        ['#FF0000', '#930031', '#930000', '#933100', '#936200'],
    'Armónica':        ['#FF0000', '#930000', '#CC0000', '#FF4F30', '#FF845C'],
    'Base':            ['#FF0000', '#CC0000', '#3B4CCA', '#FFDE00', '#B3A125'],
    'Complementaria':  ['#FF0000', '#930031', '#009362', '#4FC490', '#7A9300'],
    'Triádica':        ['#FF0000', '#319300', '#003193', '#C9435A', '#5E000A'],
}

categorias = ['A', 'B', 'C', 'D', 'E']
valores    = [23, 45, 56, 78, 32]


fig_paletas = make_subplots(rows=len(paletas), cols=1,
                             subplot_titles=list(paletas.keys()))

for i, (nombre, colores) in enumerate(paletas.items(), start=1):
    fig_paletas.add_trace(
        go.Bar(x=categorias, y=valores, marker_color=colores, showlegend=False),
        row=i, col=1
    )

fig_paletas.update_layout(
    title='Paletas de Color — Pokémon',
    height=1200,
)

fig_paletas.show()

In [ ]:
#Creacion de paleta adicional para cada tipo de pokemon(colores oficiales)
TYPE_COLORS = {
    'Normal':'#A8A878',   'Fire':'#F08030',    'Water':'#6890F0',
    'Electric':'#F8D030', 'Grass':'#78C850',   'Ice':'#98D8D8',
    'Fighting':'#C03028', 'Poison':'#A040A0',  'Ground':'#E0C068',
    'Flying':'#A890F0',   'Psychic':'#F85888', 'Bug':'#A8B820',
    'Rock':'#B8A038',     'Ghost':'#705898',   'Dragon':'#7038F8',
    'Dark':'#705848',     'Steel':'#B8B8D0',   'Fairy':'#EE99AC',
}

# Tema de colores claros
BG_COLOR   = '#FFFFFF'
CARD_COLOR = '#F5F5F5'
TEXT_COLOR = '#111111'
GRID_COLOR = '#D9D9D9'
ACCENT = '#3B82F6'


In [ ]:
#Se establece tema de color claro para las graficas
import plotly.io as pio
pio.templates.default = "plotly_white"

## Graficas con plotly

### 1.Importacion de librerias

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
import numpy as np

### 2.Cargar dataset

In [ ]:
df = pd.read_csv('pokemon_complete.csv')
df['type_2'] = df['type_2'].fillna('—')

In [ ]:
#Se cambian los nombres de generaciones para que tenga la primera letra en mayuscula
gen_order = ['gen-i','gen-ii','gen-iii','gen-iv','gen-v','gen-vi','gen-vii','gen-viii','gen-ix']
gen_labels = {'gen-i':'Gen I','gen-ii':'Gen II','gen-iii':'Gen III','gen-iv':'Gen IV',
              'gen-v':'Gen V','gen-vi':'Gen VI','gen-vii':'Gen VII','gen-viii':'Gen VIII','gen-ix':'Gen IX'}
df['gen_label'] = df['generation'].map(gen_labels)

### 3.Creación de graficas

#### 3.1 Scatter: Atack vs Speed
**¿Qué muestra?**
Cada punto representa un Pokémon posicionado según su **velocidad** (eje X) y su **ataque** (eje Y).
Los puntos se colorean por tipo primario usando los colores oficiales de cada tipo.
Los Pokémon legendarios y míticos aparecen como **estrellas de mayor tamaño**.
Las líneas en los bordes (rugs) muestran la densidad de distribución en cada eje.

**Esquema de color:**
TYPE_COLORS — colores oficiales de los 18 tipos Pokémon (Normal, Fire, Water, etc.).

In [ ]:
# 1. SCATTER: Attack vs Speed
def make_scatter():
    fig = go.Figure()

    for t in sorted(df['type_1'].unique()):
        sub = df[df['type_1'] == t]
        color = TYPE_COLORS.get(t, '#888')
        fig.add_trace(go.Scatter(
            x=sub['speed'], y=sub['attack'],
            mode='markers',
            name=t,
            marker=dict(
                color=color,
                size=np.where(sub['is_legendary'] | sub['is_mythical'], 12, 7),
                symbol=np.where(sub['is_legendary'] | sub['is_mythical'], 'star', 'circle'),
                opacity=0.85,
                line=dict(width=0.5, color='rgba(255,255,255,0.3)')
            ),
            customdata=np.stack([
                sub['name'], sub['type_1'], sub['type_2'],
                sub['hp'], sub['defense'], sub['sp_attack'],
                sub['sp_defense'], sub['base_stat_total'],
                np.where(sub['is_legendary'], '⭐ Legendary', np.where(sub['is_mythical'], '✨ Mythical', ''))
            ], axis=-1),
            hovertemplate=(
                "<b>%{customdata[0]}</b> %{customdata[8]}<br>"
                "Type: %{customdata[1]} / %{customdata[2]}<br>"
                "─────────────────<br>"
                "⚔️  Attack: <b>%{y}</b>  &nbsp; 💨 Speed: <b>%{x}</b><br>"
                "❤️  HP: %{customdata[3]}  &nbsp; 🛡️ Def: %{customdata[4]}<br>"
                "✨ SpAtk: %{customdata[5]}  &nbsp; 🔮 SpDef: %{customdata[6]}<br>"
                "BST: <b>%{customdata[7]}</b><extra></extra>"
            ),
        ))

    # Rugs
    for t in sorted(df['type_1'].unique()):
        sub = df[df['type_1'] == t]
        color = TYPE_COLORS.get(t, '#888')
        fig.add_trace(go.Scatter(
            x=sub['speed'], y=[-12]*len(sub),
            mode='markers', marker=dict(symbol='line-ns', size=8, color=color, opacity=0.3,
                                         line=dict(width=1, color=color)),
            showlegend=False, hoverinfo='skip',
        ))
        fig.add_trace(go.Scatter(
            x=[-12]*len(sub), y=sub['attack'],
            mode='markers', marker=dict(symbol='line-ew', size=8, color=color, opacity=0.3,
                                         line=dict(width=1, color=color)),
            showlegend=False, hoverinfo='skip',
        ))

    fig.update_layout(
        title=dict(text='Attack vs Speed — por Tipo primario', font=dict(size=20, color=ACCENT)),
        xaxis=dict(title='Speed', range=[-20, 215], gridcolor=GRID_COLOR, zerolinecolor=GRID_COLOR, color=TEXT_COLOR),
        yaxis=dict(title='Attack', range=[-20, 200], gridcolor=GRID_COLOR, zerolinecolor=GRID_COLOR, color=TEXT_COLOR),
        plot_bgcolor=CARD_COLOR, paper_bgcolor=BG_COLOR,
        font=dict(color=TEXT_COLOR),
        legend=dict(bgcolor='rgba(0,0,0,0.3)', bordercolor=GRID_COLOR, borderwidth=1),
        margin=dict(l=60, r=20, t=60, b=60),
        height=600,
    )
    return fig
fig = make_scatter()
fig.show()

**Interpretación**

Los tipos **Fighting** y **Dragon** concentran los valores más altos de ataque, mientras que los tipos **Electric** y **Flying** dominan en velocidad.
Existe una correlación débil entre ataque y velocidad, lo que confirma que ser rápido no implica ser poderoso y viceversa.
Los legendarios y míticos (estrellas) tienden a aparecer en los extremos superiores de ambos ejes, destacando su superioridad estadística respecto al resto.

#### 3.2 HEATMAP: Promedio de stats por Tipo principal
**¿Qué muestra?**
Cada celda representa el **promedio de un stat** (HP, Attack, Defense, Sp. Atk, Sp. Def, Speed)
para todos los Pokémon de un tipo primario.
El color no refleja el valor absoluto sino el **z-score** (qué tan por encima o debajo del promedio general está ese tipo en esa estadística), lo que facilita comparar tipos entre sí.

**Esquema de color:**
Complementaria — escala de 5 colores que va de tonos oscuros (valores bajos) a amarillo intenso (valores altos), construida a partir de `paletas['Complementaria']`.


In [ ]:
# 2. HEATMAP: Promedio de stats por Tipo principal
def make_heatmap():
    stats = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']
    stat_labels = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']

    type_avg = df.groupby('type_1')[stats].mean().round(1)
    type_avg = type_avg.sort_values('speed', ascending=False)

    z = type_avg.values
    types = list(type_avg.index)
    z_norm = (z - z.mean(axis=0)) / z.std(axis=0)

    hover = [[f"<b>{types[i]} — {stat_labels[j]}</b><br>Avg: {z[i][j]}"
              for j in range(len(stats))] for i in range(len(types))]

    # Colorscale construida desde la paleta activa
    pal = paletas['Complementaria']
    n = len(pal)
    colorscale = [[i / (n - 1), pal[i]] for i in range(n)]

    fig = go.Figure(go.Heatmap(
        z=z_norm,
        x=stat_labels,
        y=types,
        text=[[f"{v:.0f}" for v in row] for row in z],
        texttemplate="%{text}",
        textfont=dict(size=11, color=TEXT_COLOR),
        colorscale=colorscale,
        hovertext=hover,
        hovertemplate="%{hovertext}<extra></extra>",
        showscale=True,
        colorbar=dict(
            title=dict(text='Z-Score', font=dict(color=TEXT_COLOR)),
            tickfont=dict(color=TEXT_COLOR),
            bgcolor='rgba(0,0,0,0)',
        ),
    ))

    fig.update_layout(
        title=dict(text='Promedio de stats por Tipo principal (color = z-score)', font=dict(size=20, color=ACCENT)),
        xaxis=dict(side='top', tickfont=dict(size=13, color=TEXT_COLOR)),
        yaxis=dict(tickfont=dict(size=12, color=TEXT_COLOR)),
        plot_bgcolor=CARD_COLOR, paper_bgcolor=BG_COLOR,
        font=dict(color=TEXT_COLOR),
        margin=dict(l=90, r=60, t=90, b=20),
        height=620,
    )
    return fig

fig = make_heatmap()
fig.show()

**Interpretación**

Los tipos **Dragon** y **Steel** destacan con z-scores altos en varias estadísticas simultáneamente, confirmando su reputación como los tipos más equilibrados y poderosos competitivamente.
Los tipos **Bug** y **Normal** muestran valores neutros o negativos en casi todas las categorías, lo que los posiciona como los tipos estadísticamente más débiles en promedio.
La columna **Speed** presenta la mayor variación entre tipos, siendo **Flying** y **Electric** los más veloces y **Rock** el más lento.

#### 3.3 VIOLIN: Distribucion de stats base por Generacion
**¿Qué muestra?**
Cada violín representa la **distribución completa del Stat Total Base (BST)** de todos los Pokémon introducidos en una generación.
La caja interior muestra la mediana y el rango intercuartil; los puntos aislados son valores atípicos.
Permite comparar si ciertas generaciones introdujeron Pokémon más poderosos o más equilibrados que otras.

**Esquema de color:**
Complementaria — los 5 colores de `paletas['Complementaria']` se asignan cíclicamente a las 9 generaciones.


In [ ]:
# 3. VIOLIN: Distribucion de stats base por Generacion
def make_violin():
    # Paleta activa asignada a cada generación
    GEN_COLORS = {
        label: paletas['Complementaria'][i % len(paletas['Complementaria'])]
        for i, label in enumerate(gen_labels.values())
    }

    fig = go.Figure()

    for gen in gen_order:
        label = gen_labels[gen]
        sub = df[df['generation'] == gen]['base_stat_total']
        color = GEN_COLORS.get(label, '#888')

        fig.add_trace(go.Violin(
            y=sub,
            name=label,
            box_visible=True,
            meanline_visible=True,
            points='outliers',
            pointpos=0,
            line_color=color,
            opacity=0.75,
            marker=dict(color=color, size=4, opacity=0.5),
            hovertemplate=(
                f"<b>{label}</b><br>"
                "BST: <b>%{y}</b><extra></extra>"
            ),
        ))

    fig.update_layout(
        title=dict(text='Distribucion de stats base por Generacion', font=dict(size=20, color=ACCENT)),
        yaxis=dict(title='Base Stat Total', gridcolor=GRID_COLOR, color=TEXT_COLOR),
        xaxis=dict(color=TEXT_COLOR),
        plot_bgcolor=CARD_COLOR, paper_bgcolor=BG_COLOR,
        font=dict(color=TEXT_COLOR),
        showlegend=False,
        violinmode='overlay',
        margin=dict(l=60, r=20, t=60, b=40),
        height=550,
    )
    return fig

fig = make_violin()
fig.show()

**Interpretación**

La mayoría de las generaciones comparten una mediana de BST similar (~400–430), lo que sugiere un balance general consistente a lo largo de la franquicia.
Las generaciones con mayor dispersión (como Gen V y Gen VIII) introdujeron tanto Pokémon muy poderosos como muy débiles en la misma entrega.
Los valores atípicos superiores en casi todas las generaciones corresponden a Pokémon legendarios con BST por encima de 600.

#### 3.4 PARALLEL COORDINATES: Promedio por tipo
**¿Qué muestra?**
Cada línea representa el **perfil estadístico promedio de un tipo primario** a lo largo de seis stats: HP, Attack, Defense, Sp. Atk, Sp. Def y Speed.
Los valores están normalizados a [0, 1] para que los ejes sean comparables entre sí.
Permite identificar qué tipos son "generalistas" (líneas planas) y cuáles son especialistas en uno o dos stats.


**Esquema de color:**
TYPE_COLORS — colores oficiales de cada tipo Pokémon, los mismos usados en el scatter.


In [ ]:
# 4. PARALLEL COORDINATES: Promedio por tipo
def make_parallel():
    stats       = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']
    stat_labels = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
    N           = len(stats)
    xs          = list(range(N))

    type_avg = df.groupby('type_1')[stats].mean().round(1).reset_index()

    # Normalizar cada stat a [0, 1]
    mins = type_avg[stats].min()
    maxs = type_avg[stats].max()
    norm = (type_avg[stats] - mins) / (maxs - mins)

    fig = go.Figure()

    # Líneas de eje verticales
    for i in range(N):
        fig.add_shape(type='line', x0=i, x1=i, y0=-0.05, y1=1.05,
                      line=dict(color=GRID_COLOR, width=1.5))

    # Una traza por tipo
    for _, row in type_avg.iterrows():
        t     = row['type_1']
        color = TYPE_COLORS.get(t, '#888')
        ys    = norm.loc[row.name, stats].tolist()
        raws  = [row[s] for s in stats]

        cd = [[lbl, f"{raw:.0f}"] for lbl, raw in zip(stat_labels, raws)]

        fig.add_trace(go.Scatter(
            x=xs, y=ys,
            mode='lines+markers',
            name=t,
            line=dict(color=color, width=2.2),
            marker=dict(color=color, size=7,
                        line=dict(width=1, color='rgba(255,255,255,0.4)')),
            customdata=cd,
            hovertemplate=(
                f"<b style='color:{color}'>{t}</b><br>"
                "%{customdata[0]}: <b>%{customdata[1]}</b>"
                "<extra></extra>"
            ),
            hoverlabel=dict(bgcolor=CARD_COLOR, bordercolor=color,
                            font=dict(color=TEXT_COLOR, size=12)),
        ))

    # Anotaciones: nombre del eje arriba, min/max
    annotations = []
    for i, (lbl, s) in enumerate(zip(stat_labels, stats)):
        annotations.append(dict(
            x=i, y=1.13, xref='x', yref='paper',
            text=f"<b>{lbl}</b>", showarrow=False,
            font=dict(size=13, color=ACCENT), xanchor='center',
        ))
        annotations.append(dict(
            x=i, y=1.06, xref='x', yref='paper',
            text=f"{maxs[s]:.0f}", showarrow=False,
            font=dict(size=9, color='rgba(255,255,255,0.45)'), xanchor='center',
        ))
        annotations.append(dict(
            x=i, y=-0.06, xref='x', yref='paper',
            text=f"{mins[s]:.0f}", showarrow=False,
            font=dict(size=9, color='rgba(255,255,255,0.45)'), xanchor='center',
        ))

    fig.update_layout(
        title=dict(
        text='Stats promedio por Tipo',
        font=dict(size=20, color=ACCENT),
        y=0.96,
        x=0.03,
        xanchor='left'
        ),
        xaxis=dict(
            tickvals=xs, showticklabels=False,
            range=[-0.3, N - 0.7],
            showgrid=False, zeroline=False,
        ),
        yaxis=dict(
            range=[-0.12, 1.12],
            showgrid=False, zeroline=False,
            showticklabels=False,
        ),
        annotations=annotations,
        plot_bgcolor=CARD_COLOR,
        paper_bgcolor=BG_COLOR,
        font=dict(color=TEXT_COLOR),
        legend=dict(
            bgcolor='rgba(0,0,0,0.35)', bordercolor=GRID_COLOR, borderwidth=1,
            font=dict(size=10), x=1.01, y=0.5, xanchor='left', yanchor='middle',
        ),
        hovermode='closest',
        margin=dict(l=40, r=130, t=110, b=40),
        height=580,
    )
    return fig
fig = make_parallel()
fig.show()

**Interpretación**

Los tipos **Dragon** y **Psychic** muestran líneas elevadas en casi todos los ejes, confirmándolos como los tipos más completos estadísticamente.
Los tipos **Ghost** y **Fairy** presentan perfiles irregulares con picos en ataque especial pero caídas en defensa física.
El tipo **Rock** destaca por su alta defensa pero tiene los valores más bajos en velocidad, reflejando su rol como tipo lento pero resistente.

#### 3.5 SPEED TIER(Barras): Top 40 pokemon mas rapidos
**¿Qué muestra?**
Las 40 barras horizontales representan los **Pokémon con mayor velocidad base** del juego, ordenados de mayor a menor.
Cada barra se colorea según el tipo primario del Pokémon.
Las líneas verticales punteadas marcan umbrales estratégicos relevantes en el juego competitivo (Base 100, 110, 120, 130).

**Esquema de color:**
TYPE_COLORS — colores oficiales de cada tipo Pokémon, igual que en el scatter y las coordenadas paralelas.


In [ ]:
# 5. SPEED TIER: Top 40 pokemon mas rapidos
def make_speed_tier():
    top = df.nlargest(40, 'speed').copy().reset_index(drop=True)
    top['color'] = top['type_1'].map(TYPE_COLORS)
    top['marker_symbol'] = top.apply(
        lambda r: 'star' if r['is_legendary'] else ('diamond' if r['is_mythical'] else 'circle'), axis=1)
    top['label'] = top.apply(
        lambda r: '⭐' if r['is_legendary'] else ('✨' if r['is_mythical'] else ''), axis=1)

    # Umbrales de velocidad
    tier_lines = {
        'Scarf Base 100 (133)': 133,
        '+1 Base 100 (150)': 150,
        'Base 110': 110,
        'Base 130': 130,
        'Base 150': 150,
    }

    fig = go.Figure()

    # Lineas de referencia
    ref_speeds = [100, 110, 120, 130]
    for spd in ref_speeds:
        fig.add_vline(x=spd, line_dash='dot', line_color='rgba(255,255,255,0.15)',
                      annotation_text=f"Base {spd}", annotation_font_color='rgba(255,255,255,0.35)',
                      annotation_position='top right')

    # Barras
    fig.add_trace(go.Bar(
        x=top['speed'],
        y=top['name'],
        orientation='h',
        marker=dict(
            color=top['color'],
            line=dict(width=0.5, color='rgba(255,255,255,0.15)'),
            opacity=0.9,
        ),
        customdata=np.stack([
            top['name'], top['type_1'], top['type_2'],
            top['attack'], top['sp_attack'], top['defense'],
            top['sp_defense'], top['hp'], top['base_stat_total'],
            top['label'],
        ], axis=-1),
        hovertemplate=(
            "<b>%{customdata[0]}</b> %{customdata[9]}<br>"
            "Type: %{customdata[1]} / %{customdata[2]}<br>"
            "─────────────────<br>"
            "💨 Speed: <b>%{x}</b><br>"
            "⚔️  Atk: %{customdata[3]}  &nbsp; ✨ SpAtk: %{customdata[4]}<br>"
            "🛡️  Def: %{customdata[5]}  &nbsp; 🔮 SpDef: %{customdata[6]}<br>"
            "❤️  HP: %{customdata[7]}  &nbsp; BST: <b>%{customdata[8]}</b>"
            "<extra></extra>"
        ),
        text=top['speed'].astype(str),
        textposition='outside',
        textfont=dict(size=10, color=TEXT_COLOR),
    ))

    fig.update_layout(
        title=dict(text='Top 40 pokemon mas rapidos', font=dict(size=20, color=ACCENT)),
        xaxis=dict(title='Velocidad base', gridcolor=GRID_COLOR, color=TEXT_COLOR, range=[0, 230]),
        yaxis=dict(autorange='reversed', tickfont=dict(size=11), color=TEXT_COLOR),
        plot_bgcolor=CARD_COLOR, paper_bgcolor=BG_COLOR,
        font=dict(color=TEXT_COLOR),
        showlegend=False,
        margin=dict(l=160, r=60, t=60, b=40),
        height=900,
    )
    return fig

fig = make_speed_tier()
fig.show()

**Interpretación**
Los tipos **Electric**, **Flying** y **Psychic** dominan el top de velocidad, lo que es coherente con el diseño de esos tipos como ofensivos y rápidos.
La mayoría de los Pokémon del top 40 superan el umbral de **Base 110**, el punto de corte relevante en el juego competitivo para "ganar turno" frente a la mayoría de rivales.
Varios legendarios y míticos (marcados con ⭐ y ✨) aparecen en las posiciones más altas, subrayando que el poder extremo y la velocidad extrema suelen coexistir en los Pokémon más raros.

#### 3.6  Scatter Ofensivo/Defensivo con Cuadrantes

**¿Qué muestra?**
Este scatter plot enfrenta el poder ofensivo (promedio de `attack` + `sp_attack`)
contra el poder defensivo (promedio de `defense` + `sp_defense`) de cada Pokémon.
El espacio se divide en **4 cuadrantes estratégicos**:

-  **Sweeper** — alto ataque, alta defensa: Pokémon versátiles y dominantes
-  **Glass Cannon** — alto ataque, baja defensa: golpean fuerte pero son frágiles
-  **Wall** — bajo ataque, alta defensa: tanques difíciles de derribar
-  **Pivot** — bajo ataque, baja defensa: de soporte o uso situacional

**Esquema de color:** Análogos



In [ ]:
df['poder_ofensivo']  = (df['attack'] + df['sp_attack']) / 2
df['poder_defensivo'] = (df['defense'] + df['sp_defense']) / 2

med_atk = df['poder_ofensivo'].median()
med_def = df['poder_defensivo'].median()

df['cuadrante'] = np.where(
    (df['poder_ofensivo'] >= med_atk) & (df['poder_defensivo'] >= med_def), 'Sweeper',
    np.where(
        (df['poder_ofensivo'] >= med_atk) & (df['poder_defensivo'] < med_def), 'Glass Cannon',
        np.where(
            (df['poder_ofensivo'] < med_atk) & (df['poder_defensivo'] >= med_def), 'Wall',
            'Pivot'
        )
    )
)

color_map_g6 = {
    'Sweeper':      paletas['Análogos'][0],
    'Glass Cannon': paletas['Análogos'][1],
    'Wall':         paletas['Análogos'][2],
    'Pivot':        paletas['Análogos'][3],
}

fig6 = px.scatter(
    df,
    x='poder_ofensivo',
    y='poder_defensivo',
    color='cuadrante',
    color_discrete_map=color_map_g6,
    hover_name='name',
    hover_data={
        'type_1': True,
        'type_2': True,
        'poder_ofensivo': ':.1f',
        'poder_defensivo': ':.1f',
        'base_stat_total': True,
        'cuadrante': False,
    },
    title= 'Scatter Ofensivo vs Defensivo — Cuadrantes Estratégicos',
    labels={
        'poder_ofensivo':  'Poder Ofensivo (ATK + SpATK) / 2',
        'poder_defensivo': 'Poder Defensivo (DEF + SpDEF) / 2',
        'cuadrante':       'Rol Estratégico',
    },
    opacity=0.75,
)

fig6.add_vline(x=med_atk, line_dash='dash', line_color='gray', opacity=0.5)
fig6.add_hline(y=med_def, line_dash='dash', line_color='gray', opacity=0.5)

ann = dict(showarrow=False, font=dict(size=13, color='black'),
           bgcolor='rgba(0,0,0,0.15)', borderpad=5)

fig6.add_annotation(x=df['poder_ofensivo'].max()*0.93,
                    y=df['poder_defensivo'].max()*0.95, text=" Sweeper",      **ann)
fig6.add_annotation(x=df['poder_ofensivo'].max()*0.93,
                    y=df['poder_defensivo'].min()+5,    text=" Glass Cannon", **ann)
fig6.add_annotation(x=df['poder_ofensivo'].min()+2,
                    y=df['poder_defensivo'].max()*0.95, text=" Wall",         **ann)
fig6.add_annotation(x=df['poder_ofensivo'].min()+2,
                    y=df['poder_defensivo'].min()+5,    text=" Pivot",        **ann)

fig6.update_traces(marker=dict(size=6))
fig6.update_layout(legend_title_text='Rol Estratégico')
fig6.show()

**Interpretación**

La gráfica revela que la mayoría de los Pokémon caen en el cuadrante **Pivot** (bajo ataque, baja defensa), lo que sugiere que gran parte del diseño de la franquicia se enfoca en Pokémon de soporte o uso situacional.  

El cuadrante **Glass Cannon** es el segundo más poblado, confirmando que el juego favorece personajes ofensivos pero frágiles.  

Los **Sweepers** son los menos comunes, ya que combinar alto ataque y alta defensa simultáneamente es un privilegio reservado para pocos Pokémon, generalmente legendarios.

#### 3.7 Violin de capture_rate por Tier de base_stat_total

**¿Qué muestra?**
Este violin plot muestra la **distribución de la tasa de captura** (`capture_rate`)
segmentada en tres niveles de poder según el stat total base:

- **Low Tier** — `base_stat_total < 400`: Pokémon débiles, más comunes
- **Mid Tier** — `400 ≤ base_stat_total ≤ 549`: Pokémon intermedios
- **Top Tier** — `base_stat_total ≥ 550`: Pokémon poderosos, más raros

Incluye **box plot interior** y puntos de datos individuales (jitter)
para mayor detalle estadístico.

**Esquema de color:** Armónica


In [ ]:
df['habitat'] = df['habitat'].fillna('Unknown')

df['tier'] = np.where(df['base_stat_total'] < 400,  'Low Tier (<400)',
             np.where(df['base_stat_total'] <= 549, 'Mid Tier (400-549)', 'Top Tier (550+)'))

color_map_g7 = {
    'Low Tier (<400)':    paletas['Armónica'][4],
    'Mid Tier (400-549)': paletas['Armónica'][2],
    'Top Tier (550+)':    paletas['Armónica'][1],
}

fig7 = px.violin(
    df,
    x='tier',
    y='capture_rate',
    color='tier',
    color_discrete_map=color_map_g7,
    box=True,
    points='all',
    hover_name='name',
    hover_data={'capture_rate': True, 'base_stat_total': True, 'type_1': True, 'tier': False},
    title='Distribución de Tasa de Captura por Tier de Poder',
    labels={'capture_rate': 'Tasa de Captura', 'tier': 'Tier de Poder'},
    category_orders={'tier': ['Low Tier (<400)', 'Mid Tier (400-549)', 'Top Tier (550+)']},
)

for tier_name, color in color_map_g7.items():
    median_val = df[df['tier'] == tier_name]['capture_rate'].median()
    fig7.add_annotation(
        x=tier_name, y=median_val + 8,
        text=f"Md: {median_val:.0f}",
        showarrow=False,
        font=dict(size=11, color=color),
        borderpad=3,
    )

fig7.update_traces(meanline_visible=True, jitter=0.3, marker_size=3, opacity=0.8)
fig7.update_layout(showlegend=False)
fig7.show()

**Interpretación**

La distribución confirma una relación inversa clara entre poder y facilidad de captura.  

Los Pokémon de **Low Tier** presentan tasas de captura altas y muy dispersas (hasta 255), lo que los hace fáciles de atrapar.  

El **Mid Tier** muestra una distribución más concentrada en valores medios.  

El **Top Tier** se agrupa fuertemente en valores muy bajos (entre 3 y 45), lo que refleja una decisión de diseño deliberada: los Pokémon más fuertes deben ser los más difíciles de obtener, aumentando su valor y exclusividad dentro del juego.

#### 3.8 Proporción de Pokémon por Generación

**¿Qué muestra?**
Este gráfico de dona muestra **cuántos Pokémon se introdujeron en cada generación**,
permitiendo ver cómo ha crecido la Pokédex a lo largo del tiempo.
El hover muestra el número exacto y el porcentaje.

**Esquema de color:** Base


In [ ]:
gen_counts = df['generation'].value_counts().reset_index()
gen_counts.columns = ['generation', 'count']

gen_labels = {
    'gen-i':    'Gen I (Kanto)',    'gen-ii':   'Gen II (Johto)',
    'gen-iii':  'Gen III (Hoenn)',  'gen-iv':   'Gen IV (Sinnoh)',
    'gen-v':    'Gen V (Unova)',    'gen-vi':   'Gen VI (Kalos)',
    'gen-vii':  'Gen VII (Alola)',  'gen-viii': 'Gen VIII (Galar)',
    'gen-ix':   'Gen IX (Paldea)',
}
gen_counts['label'] = gen_counts['generation'].map(gen_labels)
gen_colors_list = (paletas['Base'] * 2)[:len(gen_counts)]

fig8 = go.Figure(go.Pie(
    labels=gen_counts['label'],
    values=gen_counts['count'],
    hole=0.45,
    marker=dict(colors=gen_colors_list, line=dict(color=TEXT_COLOR, width=2)),
    hovertemplate='<b>%{label}</b><br>Pokémon: %{value}<br>Porcentaje: %{percent}<extra></extra>',
    textinfo='label+percent',
    textfont_size=11,
))

fig8.update_layout(
    title='Distribución de Pokémon por Generación',
    legend=dict(orientation='v', x=1.02, y=0.5),
    annotations=[dict(text='1,350<br>Pokémon', x=0.5, y=0.5,
                      font=dict(size=16), showarrow=False)],
    width=850, height=560,
)
fig8.show()

**Interpretación**

La **Generación I (Kanto)** es la más grande de toda la Pokédex con más Pokémon introducidos, mientras que la **Generación VI (Kalos)** es la más pequeña.  

Las primeras generaciones establecieron las bases con una variedad de Pokémon, apostando por cantidad, y las generaciones más recientes han priorizado la calidad, las mecánicas nuevas y la narrativa por encima del volumen.

#### 3.9 Legendarios, Míticos, Baby y Normales por Generación

**¿Qué muestra?**
Este stacked bar chart muestra la **composición especial de cada generación**:
cuántos Pokémon normales, legendarios, míticos y baby introdujo cada una.
Permite ver qué generaciones fueron más generosas con Pokémon especiales.

**Esquema de color:** Complementaria

In [ ]:
df['categoria'] = np.where(df['is_legendary'], 'Legendario',
                  np.where(df['is_mythical'],   'Mítico',
                  np.where(df['is_baby'],        'Baby', 'Normal')))

cat_gen = df.groupby(['generation', 'categoria']).size().reset_index(name='count')

gen_labels_short = {
    'gen-i': 'Gen I', 'gen-ii': 'Gen II', 'gen-iii': 'Gen III',
    'gen-iv': 'Gen IV', 'gen-v': 'Gen V', 'gen-vi': 'Gen VI',
    'gen-vii': 'Gen VII', 'gen-viii': 'Gen VIII', 'gen-ix': 'Gen IX',
}
cat_gen['gen_label'] = cat_gen['generation'].map(gen_labels_short)

gen_order = ['Gen I','Gen II','Gen III','Gen IV','Gen V',
             'Gen VI','Gen VII','Gen VIII','Gen IX']
cat_order = ['Normal', 'Legendario', 'Mítico', 'Baby']

color_map_g9 = {
    'Normal':     paletas['Complementaria'][0],
    'Legendario': paletas['Complementaria'][1],
    'Mítico':     paletas['Complementaria'][2],
    'Baby':       paletas['Complementaria'][3],
}

fig9 = px.bar(
    cat_gen,
    x='gen_label',
    y='count',
    color='categoria',
    color_discrete_map=color_map_g9,
    category_orders={'gen_label': gen_order, 'categoria': cat_order},
    title='Composición Especial de Pokémon por Generación',
    labels={'count': 'Número de Pokémon', 'gen_label': 'Generación', 'categoria': 'Categoría'},
    text_auto=True,
    barmode='stack',
)

fig9.update_traces(
    textfont_size=10,
    hovertemplate='<b>%{x}</b><br>Categoría: %{fullData.name}<br>Cantidad: %{y}<extra></extra>',
)
fig9.update_layout(
    legend_title_text='Categoría',
    xaxis_title='Generación',
    yaxis_title='Número de Pokémon',
)
fig9.show()

**Interpretación**

La **Generación IX** fue la más generosa en Pokémon legendarios y míticos en una sola entrega, mientras que generaciones como la I y la II introdujeron relativamente pocos. Los pokemon legendarios y miticos muestra un aumento en nuevas generaciones, lo que muestra la importancia que le da la franquicia a la introduccion de legendarios.

Los Pokémon **Baby** son escasos en todas las generaciones y prácticamente desaparecen en las más recientes, sugiriendo que el diseño moderno de la franquicia se alejó de esa mecánica.

La proporción de Pokémon **Normales** sigue siendo dominante en todas las generaciones, confirmando que los especiales siempre han sido la excepción y no la regla dentro del diseño de la franquicia.

#### 3.10 Pokémon por Habitat y Tipo Primario

**¿Qué muestra?**
Este bar chart agrupado muestra **cuántos Pokémon de cada tipo primario
habitan en cada ecosistema** (grassland, mountain, sea, cave, etc.).
Permite identificar qué tipos dominan en cada habitat.

**Esquema de color:** Triádica

In [ ]:
hab_type = df[df['habitat'] != 'Unknown'].groupby(
    ['habitat', 'type_1']).size().reset_index(name='count')

top_types = df['type_1'].value_counts().head(10).index.tolist()
hab_type_filt = hab_type[hab_type['type_1'].isin(top_types)].copy()

hab_labels = {
    'grassland':     'Pradera',     'mountain':    'Montaña',
    'waters-edge':   'Orilla',      'forest':      'Bosque',
    'rough-terrain': 'T. Rugoso',   'cave':        'Cueva',
    'urban':         'Urbano',      'sea':         'Mar',
    'rare':          'Raro',
}
hab_type_filt['habitat_label'] = hab_type_filt['habitat'].map(
    lambda x: hab_labels.get(x, x))

type_list = hab_type_filt['type_1'].unique().tolist()
color_map_g10 = {t: paletas['Triádica'][i % len(paletas['Triádica'])] for i, t in enumerate(type_list)}

fig10 = px.bar(
    hab_type_filt,
    x='habitat_label',
    y='count',
    color='type_1',
    color_discrete_map=color_map_g10,
    barmode='group',
    title='Distribución de Tipos Primarios por Habitat',
    labels={'count': 'Número de Pokémon', 'habitat_label': 'Habitat', 'type_1': 'Tipo Primario'},
    hover_data={'count': True, 'type_1': True},
)

fig10.update_traces(
    hovertemplate='<b>%{x}</b><br>Tipo: %{fullData.name}<br>Pokémon: %{y}<extra></extra>',
)
fig10.update_layout(
    legend_title_text='Tipo Primario',
    xaxis=dict(title='Habitat', tickangle=-25),
    yaxis_title='Número de Pokémon',
    bargap=0.15,
    bargroupgap=0.05,
)
fig10.show()

**Interpretación**

Los habitats **Orilla (waters-edge)** y **Pradera (grassland)** concentran la mayor diversidad y cantidad de tipos, lo que los convierte en los ecosistemas más ricos del universo Pokémon.  

En contraste, habitats como **Cueva** y **Terreno Rugoso** están dominados por tipos específicos como Rock, Ground y Poison, lo cual es coherente con su entorno físico.  

El habitat **Raro** agrupa Pokémon únicos que no encajan fácilmente en ecosistemas convencionales, generalmente asociados a tipos especiales como Psychic, Dragon o Ghost.